# Outcome contrasts on the frozen samples

Compare long-term with short-term residential care using the completed imputation draws. Positive differences indicate a higher outcome rate in long-term care. These are descriptive overlap-weighted contrasts; uncertainty intervals and causal interpretation require the planned inference validation.

Dependencies: the newest S1_01 run after its six draws are verified, containing two-chain imputation draws for completion seed 722 and frequency seeds 720 and 721; numpy, pandas, and helper/run_notebook.py for execution and timed HTML export. The contrast calculation is reused from archive/S1_2/weighted_contrasts.py. No archived sample selection is applied.

In [1]:
from pathlib import Path
from datetime import datetime
import json,hashlib
import numpy as np
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'S1/helper').is_dir())
HOME=ROOT/'S1'
SPEC=json.loads((HOME/'helper/assets/sample_specification.json').read_text())
OUT=HOME/'outputs'/('S1_02_outcome_review_'+datetime.now().strftime('%Y%m%d_%H%M%S'))
def contrast(y,t,e):
    y,t,e=[np.asarray(a,dtype=float) for a in [y,t,e]]
    assert y.ndim==t.ndim==e.ndim==1 and len(y)==len(t)==len(e)>0
    assert np.isin(y,[0,1]).all() and set(t)=={0,1}
    assert np.isfinite(e).all() and ((e>0)&(e<1)).all()
    w=np.where(t==1,1-e,e)
    means=[np.average(y[t==arm],weights=w[t==arm]) for arm in [0,1]]
    return means[0],means[1],means[1]-means[0]
assert contrast([0,1,0,1],[0,1,0,1],[.5]*4)==(0,1,1)
assert contrast([1,1,1,1],[0,1,0,1],[.1,.3,.6,.8])==(1,1,0)


## Estimates, geographic concentration and sensitivity

Separate weight normalization in each treatment arm defines the risk difference. The effective number of states is the inverse sum of squared state weight shares; it describes concentration, not degrees of freedom for a significance test. Leave-one-state-out and discharge-year contrasts retain the full-sample propensity scores and weights; each subset changes the population being described. Differences between imputation draws and sensitivity rows are not confidence intervals.

In [2]:
folders=sorted((HOME/'outputs').glob('S1_01_data_preparation_*'))
if not folders:raise FileNotFoundError('Run S1_01 before S1_02: no preparation output exists')
source_run=folders[-1]
if not (source_run/'manifest.json').is_file():raise FileNotFoundError(f'Run S1_01 to completion before S1_02: {source_run.name} has no manifest')
manifest=json.loads((source_run/'manifest.json').read_text())
if manifest['status']!='verified_samples_and_imputation_draws':raise FileNotFoundError(f'Run S1_01 to completion before S1_02: {source_run.name} has no verified draws')
runs={'completion':[source_run/'completion_imputation_seed722'], 'frequency':[source_run/'frequency_imputation_seed720',source_run/'frequency_imputation_seed721']}
if not all((folder/f'engineering_imputation_chain_{chain}.csv.gz').is_file() for folders in runs.values() for folder in folders for chain in (1,2)):raise FileNotFoundError(f'Missing imputation draws in {source_run}')
print('Selected S1_01 run:',source_run.name)
rows=[];states=[];sensitivity=[]
for outcome,folders in runs.items():
    ids=None
    for run,folder in enumerate(folders,1):
        for chain in [1,2]:
            d=pd.read_csv(folder/f'engineering_imputation_chain_{chain}.csv.gz',index_col='episode_source_id')
            assert d.index.is_unique and len(d)=={'completion':241118,'frequency':146194}[outcome]
            frozen=SPEC['frozen_eligible_samples']['outcomes'][outcome]
            membership=hashlib.sha256(('\n'.join(sorted(d.index))+'\n').encode()).hexdigest()
            assert len(d)==frozen['n'] and membership==frozen['membership_sha256']
            np.testing.assert_allclose(d.overlap_weight,np.where(d.treatment.eq(1),1-d.propensity,d.propensity),rtol=0,atol=1e-10)
            if ids is None:ids=d.index
            else:assert ids.equals(d.index)
            short,long,diff=contrast(d.outcome,d.treatment,d.propensity)
            rows.append(dict(outcome=outcome,run=run,source=str(folder.relative_to(HOME/'outputs')),chain=chain,n=len(d),short_percent=100*short,long_percent=100*long,difference_pp=100*diff))
            assert d.DISYR.isin([2022,2023]).all()
            for state in sorted(d.STFIPS.unique()):
                keep=d.STFIPS.ne(state)
                s,l,delta=contrast(d.loc[keep,'outcome'],d.loc[keep,'treatment'],d.loc[keep,'propensity'])
                omitted=d.STFIPS.eq(state)
                sensitivity.append(dict(outcome=outcome,run=run,source=str(folder.relative_to(HOME/'outputs')),chain=chain,analysis='leave_one_state_out',excluded_state=int(state),discharge_year=None,n=int(keep.sum()),short_percent=100*s,long_percent=100*l,difference_pp=100*delta,full_difference_pp=100*diff,change_pp=100*(delta-diff),omitted_short_weight_share_percent=100*d.loc[omitted & d.treatment.eq(0),'overlap_weight'].sum()/d.loc[d.treatment.eq(0),'overlap_weight'].sum(),omitted_long_weight_share_percent=100*d.loc[omitted & d.treatment.eq(1),'overlap_weight'].sum()/d.loc[d.treatment.eq(1),'overlap_weight'].sum()))
            for year in [2022,2023]:
                keep=d.DISYR.eq(year)
                s,l,delta=contrast(d.loc[keep,'outcome'],d.loc[keep,'treatment'],d.loc[keep,'propensity'])
                sensitivity.append(dict(outcome=outcome,run=run,source=str(folder.relative_to(HOME/'outputs')),chain=chain,analysis='discharge_year',excluded_state=None,discharge_year=year,n=int(keep.sum()),short_percent=100*s,long_percent=100*l,difference_pp=100*delta,full_difference_pp=100*diff,change_pp=100*(delta-diff),omitted_short_weight_share_percent=None,omitted_long_weight_share_percent=None))
            for arm in [0,1]:
                g=d.loc[d.treatment.eq(arm)]
                totals=g.groupby('STFIPS').overlap_weight.sum();shares=totals/totals.sum()
                states.append(dict(outcome=outcome,run=run,source=str(folder.relative_to(HOME/'outputs')),chain=chain,treatment=arm,states=len(shares),largest_state=int(shares.idxmax()),largest_weight_share_percent=100*shares.max(),effective_states=1/(shares**2).sum()))
result=pd.DataFrame(rows);concentration=pd.DataFrame(states);sensitivity=pd.DataFrame(sensitivity)
display(result);display(concentration)
display(sensitivity.groupby(['outcome','analysis']).difference_pp.agg(['min','max']))
display(sensitivity.loc[sensitivity.analysis.eq('discharge_year'),['outcome','run','chain','discharge_year','n','difference_pp','change_pp']])
OUT.mkdir()
result.to_csv(OUT/'weighted_outcome_draws.csv',index=False)
concentration.to_csv(OUT/'state_concentration.csv',index=False)
sensitivity.to_csv(OUT/'geographic_year_sensitivity.csv',index=False)
review_manifest={'status':'descriptive_review_complete_inference_pending',
                 'source_run':source_run.name,
                 'source_manifest_sha256':hashlib.sha256((source_run/'manifest.json').read_bytes()).hexdigest(),
                 'outputs':{name:hashlib.sha256((OUT/name).read_bytes()).hexdigest() for name in
                            ['weighted_outcome_draws.csv','state_concentration.csv','geographic_year_sensitivity.csv']},
                 'role':'descriptive outcome and geographic review; no confidence interval or causal claim'}
(OUT/'manifest.json').write_text(json.dumps(review_manifest,indent=2)+'\n')
print('Output directory:',OUT)


Selected S1_01 run: S1_01_data_preparation_20260926_163212_739830
Output directory: /Users/researcher/code_projects/Causal Discovery /code/project_code/notebooks/code_0927/S1/outputs/S1_02_outcome_review_20260926_175955


,outcome,run,source,chain,n,short_percent,long_percent,difference_pp
0,completion,1,S1_01_data_preparation_20260926_163212_739830/...,1,241118,61.554040,46.537393,-15.016647
1,completion,1,S1_01_data_preparation_20260926_163212_739830/...,2,241118,61.553256,46.543254,-15.010002
2,frequency,1,S1_01_data_preparation_20260926_163212_739830/...,1,146194,61.585465,56.717891,-4.867573
3,frequency,1,S1_01_data_preparation_20260926_163212_739830/...,2,146194,61.592525,56.684078,-4.908447
4,frequency,2,S1_01_data_preparation_20260926_163212_739830/...,1,146194,61.589270,56.712822,-4.876448
5,frequency,2,S1_01_data_preparation_20260926_163212_739830/...,2,146194,61.611052,56.701526,-4.909526


,outcome,run,source,chain,treatment,states,largest_state,largest_weight_share_percent,effective_states
0,completion,1,S1_01_data_preparation_20260926_163212_739830/...,1,0,18,36,39.929177,4.774867
1,completion,1,S1_01_data_preparation_20260926_163212_739830/...,1,1,18,36,39.944700,4.770359
2,completion,1,S1_01_data_preparation_20260926_163212_739830/...,2,0,18,36,39.927642,4.774315
3,completion,1,S1_01_data_preparation_20260926_163212_739830/...,2,1,18,36,39.944547,4.769559
4,frequency,1,S1_01_data_preparation_20260926_163212_739830/...,1,0,15,36,63.089419,2.424037
5,frequency,1,S1_01_data_preparation_20260926_163212_739830/...,1,1,15,36,63.103927,2.423195
6,frequency,1,S1_01_data_preparation_20260926_163212_739830/...,2,0,15,36,63.079152,2.424861
7,frequency,1,S1_01_data_preparation_20260926_163212_739830/...,2,1,15,36,63.106611,2.423078
8,frequency,2,S1_01_data_preparation_20260926_163212_739830/...,1,0,15,36,63.070553,2.425534
9,frequency,2,S1_01_data_preparation_20260926_163212_739830/...,1,1,15,36,63.100806,2.423539


min        max
outcome    analysis                                 
completion discharge_year      -15.672885 -14.428212
           leave_one_state_out -15.817478 -12.670811
frequency  discharge_year       -8.776240  -1.753794
           leave_one_state_out  -5.734029   5.715980

,outcome,run,chain,discharge_year,n,difference_pp,change_pp
18,completion,1,1,2022.0,120472,-14.447298,0.569349
19,completion,1,1,2023.0,120646,-15.665061,-0.648414
38,completion,1,2,2022.0,120472,-14.428212,0.581791
39,completion,1,2,2023.0,120646,-15.672885,-0.662883
55,frequency,1,1,2022.0,72873,-1.753794,3.113780
56,frequency,1,1,2023.0,73321,-8.761464,-3.893891
72,frequency,1,2,2022.0,72873,-1.825917,3.082529
73,frequency,1,2,2023.0,73321,-8.765577,-3.857130
89,frequency,2,1,2022.0,72873,-1.759474,3.116974
90,frequency,2,1,2023.0,73321,-8.776240,-3.899792
